# CADENCE — Interactive Notebook GUI
**C2 Anomaly Detection via Ensemble Network Correlation Evidence**

Run each cell top-to-bottom. The full GUI renders in the final cell — no separate server needed.

---

In [ ]:
# ── Dependency check ─────────────────────────────────────────────────────
import subprocess, sys, importlib

_REQUIRED = {
    'pandas':      ('pandas',       'pandas>=2.0'),
    'numpy':       ('numpy',        'numpy>=1.24'),
    'sklearn':     ('scikit-learn',  'scikit-learn>=1.3'),
    'scipy':       ('scipy',        'scipy>=1.10'),
    'statsmodels': ('statsmodels',  'statsmodels>=0.14'),
    'matplotlib':  ('matplotlib',   'matplotlib>=3.7'),
    'pyarrow':     ('pyarrow',      'pyarrow>=14.0'),
    'shap':        ('shap',         'shap>=0.44'),
    'ipywidgets':  ('ipywidgets',   'ipywidgets>=8.0'),
}

missing, pins = [], []
for imp_name, (pkg, pin) in _REQUIRED.items():
    try:
        importlib.import_module(imp_name.split('.')[0])
    except ImportError:
        missing.append(pkg); pins.append(pin)

from pathlib import Path
_cadence_ok = (Path('.').resolve() / 'analytic_pipeline' / '__init__.py').exists()

if not _cadence_ok:
    print('\u274c analytic_pipeline not found. Run this notebook from the CADENCE repo root.')

if missing:
    print(f'\u26a0\ufe0f  Missing {len(missing)} package(s): {", ".join(missing)}')
    print('Installing...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pins)
    print('\u2705 Installed successfully \u2014 restart kernel if imports still fail.')
else:
    print('\u2705 All dependencies present')

if _cadence_ok:
    print('\u2705 analytic_pipeline found')

In [ ]:
import json, logging, sys, tempfile, threading, time, io, dataclasses
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ipywidgets as W
from IPython.display import display, HTML, clear_output

REPO_ROOT = Path('.').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from analytic_pipeline import BDPConfig, BDPPipeline
from analytic_pipeline.config import (
    IOConfig, FeatureConfig, ScalingConfig, IsolationConfig,
    PairConfig, SAXConfig, PeriodicityConfig, PELTConfig,
    CorroborationConfig, TLSCorroborationConfig, PrefilterConfig, TriageConfig,
)
from analytic_pipeline.generate_synthetic_data import SyntheticDataGenerator, evaluate_detection
print('\u2705 Imports OK')

In [ ]:
display(HTML("""
<style>
@import url('https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@400;600&family=IBM+Plex+Sans:wght@400;600&display=swap');
.cad-hdr  { font-family:'IBM Plex Mono',monospace; color:#58a6ff; font-size:1.3rem; font-weight:700; }
.cad-sec  { font-family:'IBM Plex Mono',monospace; font-size:0.7rem; color:#8b949e; text-transform:uppercase; letter-spacing:1.5px; border-bottom:1px solid #30363d; padding-bottom:4px; margin:14px 0 10px; }
.cad-card { background:#161b22; border:1px solid #21262d; border-radius:8px; padding:14px 16px; margin:8px 0; }
.cad-met  { display:inline-block; background:#161b22; border:1px solid #21262d; border-radius:6px; padding:8px 14px; margin:4px; min-width:100px; text-align:center; }
.cad-met .val { font-family:'IBM Plex Mono',monospace; font-size:1.2rem; color:#58a6ff; font-weight:700; }
.cad-met .lbl { font-family:'IBM Plex Sans',sans-serif; font-size:0.7rem; color:#8b949e; }
.cad-log  { font-family:'IBM Plex Mono',monospace; font-size:0.75rem; background:#010409; color:#58a6ff; border:1px solid #21262d; border-radius:6px; padding:10px 12px; max-height:220px; overflow-y:auto; white-space:pre-wrap; line-height:1.55; }
.cad-tbl  { font-family:'IBM Plex Mono',monospace; font-size:0.78rem; border-collapse:collapse; width:100%; }
.cad-tbl th { background:#161b22; color:#8b949e; padding:7px 10px; text-align:left; border-bottom:1px solid #21262d; font-size:0.7rem; }
.cad-tbl td { padding:7px 10px; border-bottom:1px solid #21262d; color:#c9d1d9; }
.cad-tbl tr:nth-child(even) td { background:#0f1318; }
.sc-hi { color:#f85149; font-weight:700; } .sc-md { color:#d29922; font-weight:700; } .sc-lo { color:#3fb950; font-weight:700; }
.dst-ip { color:#f85149; } .per-v { color:#58a6ff; } .mit-v { color:#484f58; font-size:0.7rem; }
.gt-ok { color:#3fb950; } .gt-ms { color:#f85149; } .gt-dc { color:#8b949e; }
.shap-row { display:flex; align-items:center; gap:8px; margin-bottom:5px; }
.shap-f { font-family:'IBM Plex Mono',monospace; font-size:0.72rem; color:#8b949e; width:170px; text-align:right; flex-shrink:0; }
.shap-t { flex:1; height:12px; background:#0d1117; border-radius:3px; overflow:hidden; }
.shap-b { height:100%; background:linear-gradient(90deg,#1f6feb,#58a6ff); border-radius:3px; }
.shap-v { font-family:'IBM Plex Mono',monospace; font-size:0.72rem; color:#58a6ff; width:36px; text-align:right; }
</style>
"""))
display(HTML('<div class="cad-hdr">CADENCE</div><div style="font-family:IBM Plex Mono,monospace;font-size:0.65rem;color:#484f58;letter-spacing:2px">C2 ANOMALY DETECTION VIA ENSEMBLE NETWORK CORRELATION EVIDENCE</div><hr style="border-color:#21262d;margin:8px 0">'))
print('Styling loaded')

In [ ]:
# Pipeline state shared between UI thread and background thread
_st = dict(running=False, logs='', complete=False, artifacts=None, syn_eval=None, labels=None, conn=None, report_html=None)
_lk = threading.Lock()

class _WLH(logging.Handler):
    def emit(self, record):
        with _lk:
            _st['logs'] += self.format(record) + '\n'

_h = _WLH()
_h.setFormatter(logging.Formatter('%(asctime)s  %(levelname)-8s  %(name)s  %(message)s', '%H:%M:%S'))
logging.getLogger().addHandler(_h)
logging.getLogger().setLevel(logging.INFO)
print('Logging configured')

In [ ]:
S = {'description_width': '220px'}
L = W.Layout(width='480px')
H = W.Layout(width='240px')

def sec(t): return W.HTML(f'<div class="cad-sec">{t}</div>')

# ── Input mode ────────────────────────────────────────────────────────────
w_mode = W.RadioButtons(
    options=['Synthetic data', 'Separate files', 'Unified file'],
    value='Synthetic data', description='Input mode:',
    style=S, layout=W.Layout(width='500px'),
)
w_mode_help = W.HTML(
    '<div style="font-size:0.75rem;color:#8b949e;margin-bottom:8px">'
    '<b>Synthetic:</b> generate test data in-memory (no files needed). '
    '<b>Unified:</b> one CSV/Parquet with a <code>log_type</code> column. '
    '<b>Separate:</b> individual files per log type.</div>'
)

# ── Synthetic params ──────────────────────────────────────────────────────
w_days  = W.IntSlider(value=30, min=3, max=90, step=1, description='Days:', style=S, layout=L)
w_bgr   = W.IntSlider(value=10000, min=1000, max=50000, step=1000, description='BG rows/day:', style=S, layout=L)
w_noisy = W.IntSlider(value=500, min=100, max=5000, step=100, description='Noisy rows/day:', style=S, layout=L)
w_seed  = W.IntText(value=42, description='Seed:', style=S, layout=H)
syn_box = W.VBox([sec('Synthetic Data'), w_days, w_bgr, w_noisy, w_seed])

# ── File uploads ──────────────────────────────────────────────────────────
w_conn = W.FileUpload(description='Conn log:', accept='.csv,.parquet,.pq', multiple=False, layout=L)
w_dns  = W.FileUpload(description='DNS log:',  accept='.csv,.parquet,.pq', multiple=False, layout=L)
w_http = W.FileUpload(description='HTTP log:', accept='.csv,.parquet,.pq', multiple=False, layout=L)
w_ssl  = W.FileUpload(description='SSL log:',  accept='.csv,.parquet,.pq', multiple=False, layout=L)
w_uni  = W.FileUpload(description='Unified:',  accept='.csv,.parquet,.pq', multiple=False, layout=L)
sep_box = W.VBox([sec('Separate Files'), w_conn, w_dns, w_http, w_ssl])
uni_box = W.VBox([sec('Unified File (log_type column)'), w_uni])

# ── I/O Config ────────────────────────────────────────────────────────────
w_io_output_dir  = W.Text(value='results', description='Output directory:', style=S, layout=L)
w_io_table_name  = W.Text(value='Conn_logs', description='Table name:', style=S, layout=L)
w_io_query_start = W.Text(value='2025-10-22 00:00:00', description='Query start (UTC):', style=S, layout=L)
w_io_query_end   = W.Text(value='2025-10-22 23:59:59', description='Query end (UTC):', style=S, layout=L)
w_io_query_limit = W.IntText(value=1_000_000, description='ISF query limit:', style=S, layout=L)
w_io_debug       = W.Checkbox(value=False, description='Debug mode', style=S)

# ── Features Config ───────────────────────────────────────────────────────
_default_keep = 'timestamp, datetime, src_ip, src_p, src_pkts, dst_ip, dst_p, resp_pkts, duration, conn_state, service, total_bytes, sin_time, cos_time, hour, minute, scenario'
w_feat_keep = W.Textarea(value=_default_keep, description='keep_cols:', style=S, layout=W.Layout(width='480px', height='60px'))

# ── Scaling Config ────────────────────────────────────────────────────────
w_scl_thresh     = W.FloatSlider(value=1.0, min=0.0, max=10.0, step=0.1, description='threshold:', style=S, layout=L)
w_scl_bin_thresh = W.FloatSlider(value=0.001, min=0.0, max=0.1, step=0.001, readout_format='.3f', description='binary_threshold:', style=S, layout=L)
w_scl_skew       = W.FloatSlider(value=2.0, min=0.0, max=10.0, step=0.1, description='skew_threshold:', style=S, layout=L)
w_scl_rr         = W.FloatSlider(value=100.0, min=1.0, max=1000.0, step=1.0, description='range_ratio_thresh:', style=S, layout=L)
w_scl_min_uniq   = W.IntSlider(value=10, min=2, max=100, step=1, description='min_unique:', style=S, layout=L)
w_scl_min_max    = W.FloatSlider(value=100.0, min=1.0, max=1000.0, step=1.0, description='min_max_threshold:', style=S, layout=L)

# ── Isolation Forest ──────────────────────────────────────────────────────
w_n_est   = W.IntSlider(value=200, min=10, max=2000, step=50, description='n_estimators:', style=S, layout=L)
w_samp    = W.IntSlider(value=3000, min=100, max=50000, step=500, description='max_samples:', style=S, layout=L)
w_contam  = W.FloatSlider(value=0.05, min=0.001, max=0.20, step=0.005, readout_format='.3f', description='contamination:', style=S, layout=L)
w_rs      = W.IntText(value=42, description='random_state:', style=S, layout=H)
w_test_sz = W.FloatSlider(value=0.3, min=0.1, max=0.5, step=0.05, description='test_size:', style=S, layout=L)
w_stab    = W.FloatSlider(value=0.80, min=0.5, max=1.0, step=0.05, description='stability_thresh:', style=S, layout=L)

# ── Pair Config ───────────────────────────────────────────────────────────
w_pair_min_obs = W.IntSlider(value=8, min=2, max=100, step=1, description='min_observations:', style=S, layout=L)
w_minflows     = W.IntSlider(value=8, min=2, max=100, step=1, description='min_pair_flows:', style=S, layout=L)
w_maxpairs     = W.IntSlider(value=5000, min=100, max=50000, step=100, description='max_pairs:', style=S, layout=L)
w_channel_key  = W.SelectMultiple(
    options=['src_ip', 'dst_ip', 'dst_port', 'proto', 'src_port'],
    value=['src_ip', 'dst_ip', 'dst_port', 'proto'],
    description='channel_key:', style=S, layout=W.Layout(width='480px', height='100px'),
)

# ── SAX Config ────────────────────────────────────────────────────────────
w_sax_word_len  = W.IntSlider(value=20, min=4, max=100, step=1, description='word_length:', style=S, layout=L)
w_sax_alpha     = W.IntSlider(value=4, min=2, max=26, step=1, description='alphabet_size:', style=S, layout=L)
w_sax_cv        = W.FloatSlider(value=0.60, min=0.0, max=2.0, step=0.05, description='cv_threshold:', style=S, layout=L)
w_sax_acf       = W.FloatSlider(value=0.30, min=0.0, max=1.0, step=0.05, description='acf_threshold:', style=S, layout=L)
w_sax_motif     = W.FloatSlider(value=0.40, min=0.0, max=1.0, step=0.05, description='motif_threshold:', style=S, layout=L)
w_sax_min_t     = W.IntSlider(value=2, min=1, max=5, step=1, description='min_tests_passing:', style=S, layout=L)
w_sax_min_obs   = W.IntSlider(value=8, min=2, max=100, step=1, description='min_observations:', style=S, layout=L)
w_sax_max_lag   = W.IntSlider(value=10, min=2, max=50, step=1, description='max_acf_lag:', style=S, layout=L)

# ── Periodicity Config ───────────────────────────────────────────────────
w_per_min_obs = W.IntSlider(value=10, min=5, max=500, step=1, description='min_observations:', style=S, layout=L)
w_nlags       = W.IntSlider(value=40, min=5, max=200, step=1, description='acf_nlags:', style=S, layout=L)
w_per_acf_sig = W.FloatSlider(value=0.25, min=0.0, max=1.0, step=0.05, description='acf_signif_thresh:', style=S, layout=L)
w_cv          = W.FloatSlider(value=0.60, min=0.0, max=2.0, step=0.05, description='cv_threshold:', style=S, layout=L)
w_per_fft     = W.FloatSlider(value=0.15, min=0.0, max=1.0, step=0.05, description='fft_power_ratio:', style=S, layout=L)
w_minper      = W.IntSlider(value=60, min=1, max=3600, step=1, description='min_period_s:', style=S, layout=L)
w_conf        = W.FloatSlider(value=0.45, min=0.0, max=1.0, step=0.05, description='confidence_thresh:', style=S, layout=L)

# ── PELT Changepoint Config ──────────────────────────────────────────────
w_pelt_penalty = W.Text(value='bic', description='penalty:', style=S, layout=H)
w_pelt_min_seg = W.IntSlider(value=5, min=2, max=100, step=1, description='min_segment_len:', style=S, layout=L)
w_pelt_min_obs = W.IntSlider(value=15, min=5, max=500, step=1, description='min_observations:', style=S, layout=L)
w_pelt_max_cp  = W.IntSlider(value=10, min=1, max=100, step=1, description='max_changepoints:', style=S, layout=L)

# ── Corroboration Config ─────────────────────────────────────────────────
w_minscore     = W.FloatSlider(value=0.55, min=0.0, max=1.0, step=0.01, readout_format='.2f', description='min_score:', style=S, layout=L)
w_pertol       = W.FloatSlider(value=0.15, min=0.0, max=0.5, step=0.01, description='period_tolerance_pct:', style=S, layout=L)
w_short_ttl    = W.IntSlider(value=300, min=10, max=3600, step=10, description='short_ttl_thresh_s:', style=S, layout=L)
w_dgaent       = W.FloatSlider(value=3.5, min=1.0, max=6.0, step=0.1, description='dga_entropy_thresh:', style=S, layout=L)
w_dga_min_len  = W.IntSlider(value=8, min=2, max=30, step=1, description='dga_min_label_len:', style=S, layout=L)
w_body_cv      = W.FloatSlider(value=0.30, min=0.0, max=2.0, step=0.05, description='http_body_cv_thresh:', style=S, layout=L)
w_uri_cv       = W.FloatSlider(value=0.40, min=0.0, max=2.0, step=0.05, description='http_uri_cv_thresh:', style=S, layout=L)
w_rarua        = W.FloatSlider(value=0.05, min=0.0, max=0.5, step=0.01, description='rare_ua_threshold:', style=S, layout=L)
w_uri_ent      = W.FloatSlider(value=4.0, min=0.0, max=8.0, step=0.1, description='uri_entropy_thresh:', style=S, layout=L)
w_nxdomain     = W.FloatSlider(value=0.10, min=0.0, max=1.0, step=0.01, readout_format='.2f', description='nxdomain_rate_thresh:', style=S, layout=L)
w_body_trim    = W.FloatSlider(value=0.05, min=0.0, max=0.2, step=0.01, readout_format='.2f', description='body_cv_trim_pct:', style=S, layout=L)
w_extra_benign = W.Text(value='', description='Extra benign suffixes:', style=S, layout=L)

# ── TLS Corroboration Config ─────────────────────────────────────────────
w_tls_sni_ent    = W.FloatSlider(value=1.0, min=0.0, max=5.0, step=0.1, description='sni_entropy_thresh:', style=S, layout=L)
w_tls_ja3_mono   = W.FloatSlider(value=0.90, min=0.0, max=1.0, step=0.05, description='ja3_monotony_thresh:', style=S, layout=L)
w_tls_cert_reuse = W.IntSlider(value=3, min=1, max=50, step=1, description='cert_reuse_min_sess:', style=S, layout=L)
w_tls_cert_age   = W.IntSlider(value=30, min=1, max=365, step=1, description='cert_age_new_days:', style=S, layout=L)
w_tls_h5_weight  = W.FloatSlider(value=0.30, min=0.0, max=1.0, step=0.05, description='h5_weight:', style=S, layout=L)
w_tls_h6_weight  = W.FloatSlider(value=0.30, min=0.0, max=1.0, step=0.05, description='h6_weight:', style=S, layout=L)
w_tls_known_c2   = W.Textarea(
    value='e7d705a3286e19ea42f587b344ee6865\n6d4e5b73a8e1c8a0f9c6e62f7b2d1a9c',
    description='Known C2 JA3:', style=S, layout=W.Layout(width='480px', height='60px'),
)

# ── Prefilter Config ─────────────────────────────────────────────────────
w_fanin  = W.FloatSlider(value=0.50, min=0.0, max=1.0, step=0.05, description='fanin_threshold:', style=S, layout=L)
w_failed = W.FloatSlider(value=0.90, min=0.0, max=1.0, step=0.05, description='failed_conn_thresh:', style=S, layout=L)

# ── Triage Config ────────────────────────────────────────────────────────
w_tri_beacon_std = W.FloatSlider(value=0.5, min=0.0, max=5.0, step=0.1, description='beaconing_std_thresh:', style=S, layout=L)
w_tri_rare_dst   = W.IntSlider(value=25, min=1, max=1000, step=1, description='rare_dst_thresh:', style=S, layout=L)
w_tri_high_vol   = W.FloatSlider(value=0.05, min=0.0, max=0.5, step=0.01, readout_format='.2f', description='high_volume_pct:', style=S, layout=L)
w_tri_off_start  = W.IntSlider(value=6, min=0, max=23, step=1, description='off_hour_start:', style=S, layout=H)
w_tri_off_end    = W.IntSlider(value=22, min=0, max=23, step=1, description='off_hour_end:', style=S, layout=H)

# ── Report / plot options ────────────────────────────────────────────────
w_report   = W.Checkbox(value=True, description='Generate HTML report', style=S)
w_suppress = W.Checkbox(value=True, description='Suppress matplotlib plots', style=S)

print('Widgets created')

In [ ]:
def build_cfg():
    '''Build a full BDPConfig from all widget values.'''
    keep_cols = tuple(s.strip() for s in w_feat_keep.value.split(',') if s.strip())
    channel_key = tuple(w_channel_key.value)
    extra_benign = tuple(s.strip() for s in w_extra_benign.value.split(',') if s.strip())
    ja3_list = tuple(s.strip() for s in w_tls_known_c2.value.splitlines() if s.strip())

    try:
        pelt_penalty = float(w_pelt_penalty.value)
    except ValueError:
        pelt_penalty = w_pelt_penalty.value

    cfg = BDPConfig(
        io=IOConfig(
            output_dir=Path(w_io_output_dir.value),
            table_name=w_io_table_name.value,
            query_start=w_io_query_start.value,
            query_end=w_io_query_end.value,
            query_limit=w_io_query_limit.value,
            debug=w_io_debug.value,
        ),
        features=FeatureConfig(keep_cols=keep_cols),
        scaling=ScalingConfig(
            threshold=w_scl_thresh.value,
            binary_threshold=w_scl_bin_thresh.value,
            skew_threshold=w_scl_skew.value,
            range_ratio_threshold=w_scl_rr.value,
            min_unique=w_scl_min_uniq.value,
            min_max_threshold=w_scl_min_max.value,
        ),
        isolation=IsolationConfig(
            n_estimators=w_n_est.value,
            max_samples=w_samp.value,
            test_size=w_test_sz.value,
            random_state=w_rs.value,
            contamination=w_contam.value,
            stability_threshold=w_stab.value,
        ),
        pair=PairConfig(
            min_observations=w_pair_min_obs.value,
            min_pair_flows=w_minflows.value,
            max_pairs=w_maxpairs.value,
            channel_key=channel_key,
        ),
        sax=SAXConfig(
            word_length=w_sax_word_len.value,
            alphabet_size=w_sax_alpha.value,
            cv_threshold=w_sax_cv.value,
            acf_threshold=w_sax_acf.value,
            motif_threshold=w_sax_motif.value,
            min_tests_passing=w_sax_min_t.value,
            min_observations=w_sax_min_obs.value,
            max_acf_lag=w_sax_max_lag.value,
        ),
        periodicity=PeriodicityConfig(
            min_observations=w_per_min_obs.value,
            acf_nlags=w_nlags.value,
            acf_significance_threshold=w_per_acf_sig.value,
            cv_threshold=w_cv.value,
            fft_power_ratio_threshold=w_per_fft.value,
            min_period_s=float(w_minper.value),
            confidence_threshold=w_conf.value,
        ),
        pelt=PELTConfig(
            penalty=pelt_penalty,
            min_segment_length=w_pelt_min_seg.value,
            min_observations=w_pelt_min_obs.value,
            max_changepoints=w_pelt_max_cp.value,
        ),
        corroboration=CorroborationConfig(
            min_score=w_minscore.value,
            period_tolerance_pct=w_pertol.value,
            short_ttl_threshold_s=float(w_short_ttl.value),
            dga_entropy_threshold=w_dgaent.value,
            dga_min_label_len=w_dga_min_len.value,
            http_body_cv_threshold=w_body_cv.value,
            http_uri_cv_threshold=w_uri_cv.value,
            rare_ua_threshold=w_rarua.value,
            uri_entropy_threshold=w_uri_ent.value,
            nxdomain_rate_threshold=w_nxdomain.value,
            body_cv_trim_pct=w_body_trim.value,
            extra_benign_domain_suffixes=extra_benign,
            tls=TLSCorroborationConfig(
                sni_entropy_threshold=w_tls_sni_ent.value,
                ja3_monotony_threshold=w_tls_ja3_mono.value,
                cert_reuse_min_sessions=w_tls_cert_reuse.value,
                cert_age_new_days=w_tls_cert_age.value,
                ja3_known_c2=ja3_list,
                h5_weight=w_tls_h5_weight.value,
                h6_weight=w_tls_h6_weight.value,
            ),
        ),
        prefilter=PrefilterConfig(
            dst_fanin_threshold=w_fanin.value,
            failed_conn_threshold=w_failed.value,
        ),
        triage=TriageConfig(
            beaconing_std_thresh=w_tri_beacon_std.value,
            rare_dst_thresh=w_tri_rare_dst.value,
            high_volume_pct=w_tri_high_vol.value,
            off_hour_range=(w_tri_off_start.value, w_tri_off_end.value),
        ),
    )
    return cfg


def _save_upload(uw):
    if not uw.value: return None
    up = list(uw.value.values())[0]
    suffix = Path(up['metadata']['name']).suffix or '.csv'
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=suffix)
    tmp.write(up['content']); tmp.flush(); tmp.close()
    return tmp.name


def run_synthetic():
    with _lk: _st.update(running=True, logs='', complete=False, artifacts=None, syn_eval=None, report_html=None)
    try:
        with _lk: _st['logs'] += f'Generating {w_days.value} days (seed={w_seed.value})...\n'
        gen = SyntheticDataGenerator(seed=w_seed.value)
        conn, dns, http, ssl, labels = gen.generate(days=w_days.value, bg_rows_per_day=w_bgr.value, noisy_rows_per_day=w_noisy.value)
        with _lk:
            _st['logs'] += f'conn:{len(conn):,}  dns:{len(dns):,}  http:{len(http):,}  ssl:{len(ssl):,}\n'
            _st['conn'] = conn; _st['labels'] = labels
        cfg = build_cfg()
        cfg.io.query_start = str(pd.to_datetime(conn['timestamp'].min(), unit='s', utc=True))[:19]
        cfg.io.query_end   = str(pd.to_datetime(conn['timestamp'].max(), unit='s', utc=True))[:19]
        tmpdir = tempfile.mkdtemp(prefix='cad_syn_')
        cfg.io.output_dir = Path(tmpdir)/'output'; cfg.io.output_dir.mkdir(parents=True, exist_ok=True)
        cfg._unified_slices = {'conn':conn,'dns':dns,'http':http,'ssl':ssl}
        cp = Path(tmpdir)/'conn.csv'; conn.to_csv(cp, index=False); cfg.io.input_csv = cp
        with _lk: _st['logs'] += 'Running pipeline...\n'
        pipe = BDPPipeline(cfg)
        if w_report.value:
            from analytic_pipeline.report import ReportContext
            rd = cfg.io.output_dir/'report'; rd.mkdir(exist_ok=True)
            with ReportContext(output_dir=rd, open_browser=False) as rpt:
                art = pipe.run(visualize=not w_suppress.value)
                rpt_path = rpt.finalise(art)
            if rpt_path and Path(rpt_path).exists():
                with _lk: _st['report_html'] = Path(rpt_path).read_text(encoding='utf-8')
        else:
            art = pipe.run(visualize=not w_suppress.value)
        syn_eval = None
        if not art.corroboration.empty:
            syn_eval = evaluate_detection(art.corroboration, labels, conn)
            p,r,f = float(syn_eval['precision'].iloc[0]),float(syn_eval['recall'].iloc[0]),float(syn_eval['f1'].iloc[0])
            with _lk: _st['logs'] += f'P={p:.3f}  R={r:.3f}  F1={f:.3f}\n'
        with _lk: _st.update(artifacts=art, syn_eval=syn_eval, complete=True); _st['logs'] += 'Pipeline complete.\n'
    except Exception as e:
        import traceback
        with _lk: _st['logs'] += f'ERROR: {e}\n{traceback.format_exc()}\n'; _st['complete']=True
    finally:
        with _lk: _st['running'] = False


def run_files():
    with _lk: _st.update(running=True, logs='', complete=False, artifacts=None, syn_eval=None, report_html=None)
    try:
        cfg = build_cfg()
        tmpdir = tempfile.mkdtemp(prefix='cad_')
        cfg.io.output_dir = Path(tmpdir)/'output'; cfg.io.output_dir.mkdir(parents=True, exist_ok=True)
        if w_mode.value == 'Unified file':
            up = _save_upload(w_uni)
            if not up: raise ValueError('No unified file uploaded')
            cfg.io.input_unified = Path(up)
            dns_p = http_p = ssl_p = None
        else:
            cp = _save_upload(w_conn)
            if not cp: raise ValueError('No conn log uploaded')
            p = Path(cp)
            if p.suffix.lower() in ('.parquet','.pq'): cfg.io.input_parquet = p
            else: cfg.io.input_csv = p
            dns_p=_save_upload(w_dns); http_p=_save_upload(w_http); ssl_p=_save_upload(w_ssl)
        with _lk: _st['logs'] += 'Running pipeline...\n'
        pipe = BDPPipeline(cfg)
        if w_report.value:
            from analytic_pipeline.report import ReportContext
            rd = cfg.io.output_dir/'report'; rd.mkdir(exist_ok=True)
            with ReportContext(output_dir=rd, open_browser=False) as rpt:
                art = pipe.run(dns_log_path=dns_p, http_log_path=http_p, ssl_log_path=ssl_p, visualize=not w_suppress.value)
                rpt_path = rpt.finalise(art)
            if rpt_path and Path(rpt_path).exists():
                with _lk: _st['report_html'] = Path(rpt_path).read_text(encoding='utf-8')
        else:
            art = pipe.run(dns_log_path=dns_p, http_log_path=http_p, ssl_log_path=ssl_p, visualize=not w_suppress.value)
        with _lk: _st.update(artifacts=art, complete=True); _st['logs'] += 'Pipeline complete.\n'
    except Exception as e:
        import traceback
        with _lk: _st['logs'] += f'ERROR: {e}\n{traceback.format_exc()}\n'; _st['complete']=True
    finally:
        with _lk: _st['running'] = False

print('Pipeline functions ready')

In [ ]:
def render_funnel(art):
    def sl(df): return len(df) if df is not None and not df.empty else 0
    def ss(df,c):
        try: return int(df[c].sum()) if df is not None and not df.empty and c in df.columns else 0
        except: return 0
    ms = [('Raw rows',sl(art.raw)),('Pre-filtered',sl(art.prefiltered)),('IForest anom.',sl(art.anomalies)),
          ("Pairs eval'd",sl(art.sax)),('SAX pass',ss(art.sax,'sax_prescreen_pass')),
          ('Beacon pairs',ss(art.periodicity,'is_beacon_pair')),('Corroborated',ss(art.corroboration,'corroborated'))]
    cards=''.join(f'<div class="cad-met"><div class="val">{v:,}</div><div class="lbl">{l}</div></div>' for l,v in ms)
    display(HTML(f'<div class="cad-sec">Pipeline Funnel</div><div>{cards}</div>'))


def render_shap(art):
    if art.shap_values is None or art.shap_values.empty: return
    sc=[c for c in art.shap_values.columns if c.startswith('shap_') and c!='shap_sum']
    if not sc: return
    # Mean |SHAP| bar chart (matplotlib)
    ma=art.shap_values[sc].abs().mean().sort_values(ascending=False)
    ma.index=[c.replace('shap_','') for c in ma.index]
    fig, ax = plt.subplots(figsize=(8, 4))
    fig.patch.set_facecolor('#0b0f17')
    ax.set_facecolor('#161b22')
    ax.barh(ma.index[::-1], ma.values[::-1], color='#1f6feb', edgecolor='#21262d', linewidth=0.5)
    ax.set_xlabel('Mean |SHAP value|', color='#8b949e', fontsize=10)
    ax.set_title('IForest Feature Importance \u2014 Mean |SHAP|', color='#58a6ff', fontsize=11, pad=10)
    ax.tick_params(colors='#8b949e', labelsize=9)
    for spine in ax.spines.values(): spine.set_edgecolor('#21262d')
    plt.tight_layout()
    display(HTML('<div class="cad-sec">SHAP \u2014 IForest Feature Importance</div>'))
    display(HTML('<div class="cad-card" style="font-size:0.72rem;color:#8b949e;margin-bottom:8px">'
                 'SHAP values explain <b>why each pair was flagged by Isolation Forest</b> \u2014 '
                 'not why it is believed to be C2. For that, see the corroboration hypotheses (H1\u2013H6). '
                 'These two signals are complementary.</div>'))
    display(fig)
    plt.close(fig)


def render_shap_waterfall(art, out_wf):
    """Build a dropdown for per-pair SHAP waterfall and render into out_wf."""
    if art.shap_values is None or art.shap_values.empty: return
    if art.corroboration is None or art.corroboration.empty: return
    sc = [c for c in art.shap_values.columns if c.startswith('shap_') and c != 'shap_sum']
    if not sc: return
    if 'channel_id' not in art.shap_values.columns: return

    corr = art.corroboration
    if 'channel_id' in corr.columns:
        available = [cid for cid in corr['channel_id'].tolist() if cid in art.shap_values['channel_id'].values]
    else:
        available = []
    if not available: return

    display(HTML('<div class="cad-sec">SHAP Waterfall \u2014 Single Pair Explanation</div>'))
    display(HTML('<div style="font-size:0.75rem;color:#8b949e;margin-bottom:8px">'
                 'Select a corroborated lead to see exactly which features drove its anomaly score.</div>'))

    dd = W.Dropdown(options=available, description='Channel:', style=S, layout=L)

    def _render_wf(change):
        pair_id = dd.value
        with out_wf:
            clear_output(wait=True)
            try:
                import shap
                row = art.shap_values[art.shap_values['channel_id'] == pair_id]
                sv = row[sc].values[0]
                feat_names = [c.replace('shap_', '') for c in sc]
                explainer = shap.TreeExplainer(art.iforest_model)
                base_val = explainer.expected_value
                if hasattr(art, 'scaled') and 'channel_id' in art.scaled.columns:
                    pair_row = art.scaled[art.scaled['channel_id'] == pair_id]
                    stdz_cols = [c for c in [f'{n}_stdz' for n in feat_names] if c in art.scaled.columns]
                    feat_vals = pair_row[stdz_cols].fillna(0).values[0] if not pair_row.empty and stdz_cols else sv
                else:
                    feat_vals = sv
                explanation = shap.Explanation(values=sv, base_values=base_val, data=feat_vals, feature_names=feat_names)
                fig2, ax2 = plt.subplots(figsize=(9, 4))
                fig2.patch.set_facecolor('#0b0f17')
                shap.plots.waterfall(explanation, show=False)
                plt.title(f'SHAP Waterfall \u2014 {pair_id}', color='#58a6ff', fontsize=11)
                plt.tight_layout()
                display(fig2)
                plt.close(fig2)
            except Exception as e:
                display(HTML(f'<div style="color:#f85149">Could not render waterfall: {e}</div>'))

    dd.observe(_render_wf, names='value')
    display(dd)
    display(out_wf)
    _render_wf(None)


def render_leads(art):
    if art.corroboration is None or art.corroboration.empty:
        display(HTML('<div class="cad-card" style="color:#8b949e">No corroborated leads. Try lowering min_score.</div>')); return
    COLS=['src_ip','dst_ip','dst_port','proto','triage_score','beacon_confidence','dominant_period_s','mitre_techniques']
    show=[c for c in COLS if c in art.corroboration.columns]
    df=art.corroboration[show].sort_values('triage_score',ascending=False).reset_index(drop=True)
    def sc(v):
        try: f=float(v); return 'sc-hi' if f>0.85 else ('sc-md' if f>0.7 else 'sc-lo')
        except: return ''
    hdr=''.join(f'<th>{c}</th>' for c in show)
    rows=''
    for _,row in df.iterrows():
        cells=''
        for col in show:
            v=str(row[col]) if row[col] is not None else ''
            if col=='triage_score': cells+=f'<td class="{sc(v)}">{v}</td>'
            elif col=='dst_ip': cells+=f'<td class="dst-ip">{v}</td>'
            elif col=='dominant_period_s': cells+=f'<td class="per-v">{v}</td>'
            elif col=='mitre_techniques': cells+=f'<td class="mit-v">{v}</td>'
            else: cells+=f'<td>{v}</td>'
        rows+=f'<tr>{cells}</tr>'
    display(HTML(f'<div class="cad-sec">Corroborated Leads</div><div style="overflow-x:auto"><table class="cad-tbl"><thead><tr>{hdr}</tr></thead><tbody>{rows}</tbody></table></div>'))


def render_gt(se):
    if se is None or se.empty: return
    p,r,f=float(se['precision'].iloc[0]),float(se['recall'].iloc[0]),float(se['f1'].iloc[0])
    def mc(lbl,v):
        g=v>=0.75; c='#3fb950' if g else '#f85149'; t='\u2265 0.75' if g else '< 0.75'
        return f'<div class="cad-met"><div class="val" style="color:{c}">{v:.3f}</div><div class="lbl">{lbl}</div><div style="font-size:0.65rem;color:{c}">{t}</div></div>'
    dc=[c for c in ['scenario','malicious','detected'] if c in se.columns]
    df=se[dc].copy()
    df['result']=df.apply(lambda r:'Detected' if r.get('detected') else ('MISSED' if r.get('malicious') else 'Correctly suppressed'),axis=1)
    def rc(r): return 'gt-ok' if 'Detected'==r else ('gt-ms' if 'MISSED'==r else 'gt-dc')
    trows=''
    for _,r in df.iterrows():
        mal=r.get('malicious', False)
        clr='#d29922' if mal else '#8b949e'
        tlbl='malicious' if mal else 'decoy'
        trows+=f'<tr><td>{r["scenario"]}</td><td style="color:{clr}">{tlbl}</td><td class="{rc(r["result"])}">{r["result"]}</td></tr>'
    display(HTML(f'<div class="cad-sec">Ground-Truth Evaluation</div><div>{mc("Precision",p)}{mc("Recall",r)}{mc("F1",f)}</div><div style="margin-top:10px;overflow-x:auto"><table class="cad-tbl"><thead><tr><th>scenario</th><th>type</th><th>result</th></tr></thead><tbody>{trows}</tbody></table></div>'))


def render_downloads(art):
    display(HTML('<div class="cad-sec">Downloads</div>'))
    saved = []
    for name,df in [('priority',art.priority),('periodicity',art.periodicity),('corroboration',art.corroboration),('shap_values',art.shap_values)]:
        if df is not None and not df.empty:
            out=Path(tempfile.gettempdir())/f'cadence_{name}.csv'; df.to_csv(out,index=False)
            saved.append(f'<div style="font-family:IBM Plex Mono,monospace;font-size:0.78rem;margin-bottom:4px">cadence_{name}.csv saved to <code>{out}</code> ({len(df):,} rows)</div>')
    with _lk:
        rpt_html = _st.get('report_html')
    if rpt_html:
        rpt_path = Path(tempfile.gettempdir())/'cadence_report.html'
        rpt_path.write_text(rpt_html, encoding='utf-8')
        saved.append(f'<div style="font-family:IBM Plex Mono,monospace;font-size:0.78rem;margin-bottom:4px">cadence_report.html saved to <code>{rpt_path}</code></div>')
    display(HTML(''.join(saved) if saved else '<div style="color:#8b949e">No artifacts to download.</div>'))

print('Rendering functions ready')

In [ ]:
out_log = W.Output()
out_res = W.Output()
out_st  = W.Output()
out_waterfall = W.Output()

prog = W.IntProgress(value=0, min=0, max=10, description='Stage 0/10', bar_style='info', layout=W.Layout(width='500px'))
prog.layout.visibility = 'hidden'

run_btn  = W.Button(description='Run Pipeline', button_style='success', icon='play', layout=W.Layout(width='180px', height='38px'))
save_btn = W.Button(description='Save config', layout=W.Layout(width='130px'))
load_btn = W.Button(description='Load config', layout=W.Layout(width='130px'))
cfg_path = W.Text(value='cadence_config.json', description='Config path:', style=S, layout=L)

def _upd_panels(change=None):
    m=w_mode.value
    syn_box.layout.display='flex' if m=='Synthetic data' else 'none'
    sep_box.layout.display='flex' if m=='Separate files' else 'none'
    uni_box.layout.display='flex' if m=='Unified file' else 'none'
w_mode.observe(_upd_panels, names='value'); _upd_panels()


def _save(b):
    d = dataclasses.asdict(build_cfg())
    with open(cfg_path.value, 'w') as f: json.dump(d, f, indent=2, default=str)
    with out_st: clear_output(); display(HTML(f'<span style="color:#3fb950;font-family:IBM Plex Mono,monospace;font-size:0.8rem">Saved to {cfg_path.value}</span>'))
save_btn.on_click(_save)


def _load(b):
    try:
        with open(cfg_path.value) as f: d = json.load(f)
        # I/O
        io_d = d.get('io', {})
        if 'output_dir' in io_d: w_io_output_dir.value = str(io_d['output_dir'])
        if 'table_name' in io_d: w_io_table_name.value = io_d['table_name']
        if 'query_start' in io_d: w_io_query_start.value = io_d['query_start']
        if 'query_end' in io_d: w_io_query_end.value = io_d['query_end']
        if 'query_limit' in io_d: w_io_query_limit.value = int(io_d['query_limit'])
        if 'debug' in io_d: w_io_debug.value = bool(io_d['debug'])
        # Features
        feat_d = d.get('features', {})
        if 'keep_cols' in feat_d: w_feat_keep.value = ', '.join(feat_d['keep_cols'])
        # Scaling
        scl = d.get('scaling', {})
        if 'threshold' in scl: w_scl_thresh.value = float(scl['threshold'])
        if 'binary_threshold' in scl: w_scl_bin_thresh.value = float(scl['binary_threshold'])
        if 'skew_threshold' in scl: w_scl_skew.value = float(scl['skew_threshold'])
        if 'range_ratio_threshold' in scl: w_scl_rr.value = float(scl['range_ratio_threshold'])
        if 'min_unique' in scl: w_scl_min_uniq.value = int(scl['min_unique'])
        if 'min_max_threshold' in scl: w_scl_min_max.value = float(scl['min_max_threshold'])
        # Isolation
        iso = d.get('isolation', {})
        if 'n_estimators' in iso: w_n_est.value = int(iso['n_estimators'])
        if 'max_samples' in iso: w_samp.value = int(iso['max_samples'])
        if 'contamination' in iso: w_contam.value = float(iso['contamination'])
        if 'random_state' in iso: w_rs.value = int(iso['random_state'])
        if 'test_size' in iso: w_test_sz.value = float(iso['test_size'])
        if 'stability_threshold' in iso: w_stab.value = float(iso['stability_threshold'])
        # Pair
        pair = d.get('pair', {})
        if 'min_observations' in pair: w_pair_min_obs.value = int(pair['min_observations'])
        if 'min_pair_flows' in pair: w_minflows.value = int(pair['min_pair_flows'])
        if 'max_pairs' in pair: w_maxpairs.value = int(pair['max_pairs'])
        if 'channel_key' in pair: w_channel_key.value = tuple(pair['channel_key'])
        # SAX
        sax = d.get('sax', {})
        if 'word_length' in sax: w_sax_word_len.value = int(sax['word_length'])
        if 'alphabet_size' in sax: w_sax_alpha.value = int(sax['alphabet_size'])
        if 'cv_threshold' in sax: w_sax_cv.value = float(sax['cv_threshold'])
        if 'acf_threshold' in sax: w_sax_acf.value = float(sax['acf_threshold'])
        if 'motif_threshold' in sax: w_sax_motif.value = float(sax['motif_threshold'])
        if 'min_tests_passing' in sax: w_sax_min_t.value = int(sax['min_tests_passing'])
        if 'min_observations' in sax: w_sax_min_obs.value = int(sax['min_observations'])
        if 'max_acf_lag' in sax: w_sax_max_lag.value = int(sax['max_acf_lag'])
        # Periodicity
        per = d.get('periodicity', {})
        if 'min_observations' in per: w_per_min_obs.value = int(per['min_observations'])
        if 'acf_nlags' in per: w_nlags.value = int(per['acf_nlags'])
        if 'acf_significance_threshold' in per: w_per_acf_sig.value = float(per['acf_significance_threshold'])
        if 'cv_threshold' in per: w_cv.value = float(per['cv_threshold'])
        if 'fft_power_ratio_threshold' in per: w_per_fft.value = float(per['fft_power_ratio_threshold'])
        if 'min_period_s' in per: w_minper.value = int(per['min_period_s'])
        if 'confidence_threshold' in per: w_conf.value = float(per['confidence_threshold'])
        # PELT
        pelt = d.get('pelt', {})
        if 'penalty' in pelt: w_pelt_penalty.value = str(pelt['penalty'])
        if 'min_segment_length' in pelt: w_pelt_min_seg.value = int(pelt['min_segment_length'])
        if 'min_observations' in pelt: w_pelt_min_obs.value = int(pelt['min_observations'])
        if 'max_changepoints' in pelt: w_pelt_max_cp.value = int(pelt['max_changepoints'])
        # Corroboration
        corr = d.get('corroboration', {})
        if 'min_score' in corr: w_minscore.value = float(corr['min_score'])
        if 'period_tolerance_pct' in corr: w_pertol.value = float(corr['period_tolerance_pct'])
        if 'short_ttl_threshold_s' in corr: w_short_ttl.value = int(corr['short_ttl_threshold_s'])
        if 'dga_entropy_threshold' in corr: w_dgaent.value = float(corr['dga_entropy_threshold'])
        if 'dga_min_label_len' in corr: w_dga_min_len.value = int(corr['dga_min_label_len'])
        if 'http_body_cv_threshold' in corr: w_body_cv.value = float(corr['http_body_cv_threshold'])
        if 'http_uri_cv_threshold' in corr: w_uri_cv.value = float(corr['http_uri_cv_threshold'])
        if 'rare_ua_threshold' in corr: w_rarua.value = float(corr['rare_ua_threshold'])
        if 'uri_entropy_threshold' in corr: w_uri_ent.value = float(corr['uri_entropy_threshold'])
        if 'nxdomain_rate_threshold' in corr: w_nxdomain.value = float(corr['nxdomain_rate_threshold'])
        if 'body_cv_trim_pct' in corr: w_body_trim.value = float(corr['body_cv_trim_pct'])
        if 'extra_benign_domain_suffixes' in corr: w_extra_benign.value = ', '.join(corr['extra_benign_domain_suffixes'])
        # TLS
        tls = corr.get('tls', {})
        if 'sni_entropy_threshold' in tls: w_tls_sni_ent.value = float(tls['sni_entropy_threshold'])
        if 'ja3_monotony_threshold' in tls: w_tls_ja3_mono.value = float(tls['ja3_monotony_threshold'])
        if 'cert_reuse_min_sessions' in tls: w_tls_cert_reuse.value = int(tls['cert_reuse_min_sessions'])
        if 'cert_age_new_days' in tls: w_tls_cert_age.value = int(tls['cert_age_new_days'])
        if 'h5_weight' in tls: w_tls_h5_weight.value = float(tls['h5_weight'])
        if 'h6_weight' in tls: w_tls_h6_weight.value = float(tls['h6_weight'])
        if 'ja3_known_c2' in tls: w_tls_known_c2.value = '\n'.join(tls['ja3_known_c2'])
        # Prefilter
        pf = d.get('prefilter', {})
        if 'dst_fanin_threshold' in pf: w_fanin.value = float(pf['dst_fanin_threshold'])
        if 'failed_conn_threshold' in pf: w_failed.value = float(pf['failed_conn_threshold'])
        # Triage
        tri = d.get('triage', {})
        if 'beaconing_std_thresh' in tri: w_tri_beacon_std.value = float(tri['beaconing_std_thresh'])
        if 'rare_dst_thresh' in tri: w_tri_rare_dst.value = int(tri['rare_dst_thresh'])
        if 'high_volume_pct' in tri: w_tri_high_vol.value = float(tri['high_volume_pct'])
        if 'off_hour_range' in tri:
            w_tri_off_start.value = int(tri['off_hour_range'][0])
            w_tri_off_end.value = int(tri['off_hour_range'][1])
        with out_st: clear_output(); display(HTML(f'<span style="color:#3fb950;font-family:IBM Plex Mono,monospace;font-size:0.8rem">Loaded {cfg_path.value}</span>'))
    except Exception as e:
        with out_st: clear_output(); display(HTML(f'<span style="color:#f85149;font-family:IBM Plex Mono,monospace;font-size:0.8rem">{e}</span>'))
load_btn.on_click(_load)

STAGE_KWS=['Stage 1','Stage 2','Stage 3','Stage 4','Stage 5','Stage 6','Stage 7','Stage 8','Stage 9','Stage 10']

def _on_run(b):
    with _lk:
        if _st['running']: return
    run_btn.disabled=True; run_btn.description='Running...'
    prog.value=0; prog.bar_style='info'; prog.layout.visibility='visible'
    with out_res: clear_output()
    with out_log: clear_output()
    t=threading.Thread(target=run_synthetic if w_mode.value=='Synthetic data' else run_files, daemon=True)
    t.start()
    def _poll():
        prev_len=0; start=time.time()
        while True:
            time.sleep(0.8)
            with _lk:
                logs=_st['logs']; done=_st['complete']; art=_st['artifacts']; se=_st['syn_eval']
            if len(logs)!=prev_len:
                prev_len=len(logs)
                with out_log: clear_output(wait=True); display(HTML(f'<div class="cad-log">{logs}</div>'))
            stage=sum(1 for kw in STAGE_KWS if kw+':' in logs or kw+' ' in logs)
            el=int(time.time()-start)
            prog.value=stage; prog.description=f'Stage {stage}/10  {el}s'
            if done: break
        with out_log: clear_output(wait=True); display(HTML(f'<div class="cad-log">{logs}</div>'))
        run_btn.disabled=False; run_btn.description='Run Pipeline'
        if art is not None:
            prog.value=10; prog.bar_style='success'; prog.description=f'Done  {int(time.time()-start)}s'
            with out_res:
                clear_output()
                render_funnel(art); render_shap(art)
                render_shap_waterfall(art, out_waterfall)
                render_leads(art)
                if se is not None: render_gt(se)
                render_downloads(art)
        else:
            prog.bar_style='danger'
    threading.Thread(target=_poll, daemon=True).start()
run_btn.on_click(_on_run)

cfg_tabs = W.Tab(children=[
    W.VBox([sec('I/O'), w_io_output_dir, w_io_table_name, w_io_query_start, w_io_query_end, w_io_query_limit, w_io_debug]),
    W.VBox([sec('Features'), w_feat_keep]),
    W.VBox([sec('Scaling / Variance Filter'), w_scl_thresh, w_scl_bin_thresh, w_scl_skew, w_scl_rr, w_scl_min_uniq, w_scl_min_max]),
    W.VBox([sec('Isolation Forest'), w_n_est, w_samp, w_contam, w_rs, w_test_sz, w_stab]),
    W.VBox([sec('Pair Config'), w_pair_min_obs, w_minflows, w_maxpairs, w_channel_key]),
    W.VBox([sec('SAX Config'), w_sax_word_len, w_sax_alpha, w_sax_cv, w_sax_acf, w_sax_motif, w_sax_min_t, w_sax_min_obs, w_sax_max_lag]),
    W.VBox([sec('Periodicity'), w_per_min_obs, w_nlags, w_per_acf_sig, w_cv, w_per_fft, w_minper, w_conf]),
    W.VBox([sec('PELT Changepoint'), w_pelt_penalty, w_pelt_min_seg, w_pelt_min_obs, w_pelt_max_cp]),
    W.VBox([sec('Corroboration'), w_minscore, w_pertol, w_short_ttl, w_dgaent, w_dga_min_len, w_body_cv, w_uri_cv, w_rarua, w_uri_ent, w_nxdomain, w_body_trim, w_extra_benign]),
    W.VBox([sec('TLS (H5/H6)'), w_tls_sni_ent, w_tls_ja3_mono, w_tls_cert_reuse, w_tls_cert_age, w_tls_h5_weight, w_tls_h6_weight, w_tls_known_c2]),
    W.VBox([sec('Prefilter'), w_fanin, w_failed]),
    W.VBox([sec('Triage / Priority'), w_tri_beacon_std, w_tri_rare_dst, w_tri_high_vol, w_tri_off_start, w_tri_off_end]),
])
for i, t in enumerate(['I/O', 'Features', 'Scaling', 'IForest', 'Pair', 'SAX', 'Periodicity', 'PELT', 'Corroboration', 'TLS', 'Prefilter', 'Triage']):
    cfg_tabs.set_title(i, t)

gui = W.VBox([
    W.HTML('<div class="cad-sec">Data Input</div>'), w_mode, w_mode_help, syn_box, sep_box, uni_box,
    W.HTML('<div class="cad-sec">Configuration</div>'), cfg_tabs,
    W.HTML('<div class="cad-sec">Config I/O</div>'), cfg_path, W.HBox([save_btn, load_btn]), out_st,
    W.HTML('<div class="cad-sec">Run</div>'), w_report, w_suppress, run_btn, prog,
    W.HTML('<div class="cad-sec">Output</div>'), out_log, out_res,
])
display(gui)